# Machine Failure Prediction — AI4I 2020

A binary classification workflow for predicting machine failures using the AI4I 2020 Predictive Maintenance dataset.

## Workflow
1. Load data and configure reproducibility.
2. Remove identifiers and post-failure reason indicators.
3. Create a stratified train/test split.
4. Train Logistic Regression and Random Forest pipelines.
5. Evaluate with stratified cross-validation.
6. Select a decision threshold from training out-of-fold predictions.
7. Evaluate on the held-out test set and inspect feature importance.


## 1. Load data and configure the environment
Load the CSV from `data/ai4i2020.csv`. Install dependencies from the repository's `requirements.txt` before running.


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from collections import defaultdict

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, precision_score,
                             recall_score, f1_score, roc_auc_score, average_precision_score,
                             precision_recall_curve)

RANDOM_STATE = 42
df = pd.read_csv('../data/ai4i2020.csv')
print(df.shape)
df.head()


## 2. Remove identifiers and leakage-prone columns
`UDI` and `Product ID` are identifiers. `TWF`, `HDF`, `PWF`, `OSF`, and `RNF` encode failure reasons and are excluded to reduce target leakage.


In [ ]:
LEAKY_COLS = ['UDI', 'Product ID', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
df1 = df.drop(columns=LEAKY_COLS)
X = df1.drop(columns=['Machine failure'])
y = df1['Machine failure']
print(X.columns.tolist())
print(y.value_counts())


## 3. Stratified train/test split
Stratification preserves the target-class ratio in both subsets. The test set is reserved for final evaluation.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print('Train:', X_train.shape, '| failures:', y_train.sum())
print('Test:', X_test.shape, '| failures:', y_test.sum())


## 4. Preprocessing and Logistic Regression baseline
Numeric features are expanded with degree-two polynomial terms and standardized. The categorical `Type` feature is one-hot encoded. All preprocessing is fitted inside the pipeline.


In [ ]:
numeric_features = ['Air temperature [K]', 'Process temperature [K]',
                    'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
categorical_features = ['Type']

num_pipe_lr = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scale', StandardScaler())
])
pre_lr = ColumnTransformer([
    ('num', num_pipe_lr, numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])
model_lr = Pipeline([
    ('pre', pre_lr),
    ('clf', LogisticRegression(max_iter=1000))
])
model_lr.fit(X_train, y_train)
prob_lr = model_lr.predict_proba(X_test)[:, 1]
pred_lr = (prob_lr >= 0.5).astype(int)
print(confusion_matrix(y_test, pred_lr))
print(classification_report(y_test, pred_lr, digits=3))


## 5. Random Forest model
The Random Forest uses the same polynomial numeric expansion and one-hot encoding. Scaling is omitted because tree-based models do not require a shared feature scale.


In [ ]:
pre_rf = ColumnTransformer([
    ('num', PolynomialFeatures(degree=2, include_bias=False), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])
model_rf = Pipeline([
    ('pre', pre_rf),
    ('clf', RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1))
])
model_rf.fit(X_train, y_train)
prob_rf = model_rf.predict_proba(X_test)[:, 1]
pred_rf = (prob_rf >= 0.5).astype(int)
print(confusion_matrix(y_test, pred_rf))
print(classification_report(y_test, pred_rf, digits=3))


## 6. Stratified cross-validation and threshold selection
Five-fold stratified cross-validation estimates variability on training data. The F1-oriented threshold is selected from training out-of-fold predictions, not from the test set.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_validate(
    model_rf, X_train, y_train, cv=cv,
    scoring=['precision', 'recall', 'f1', 'roc_auc', 'average_precision'],
    n_jobs=-1
)
pd.DataFrame(cv_scores).filter(like='test_').agg(['mean', 'std']).T


In [ ]:
oof_prob = cross_val_predict(
    model_rf, X_train, y_train, cv=cv, method='predict_proba', n_jobs=-1
)[:, 1]
thresholds = np.linspace(0.05, 0.95, 181)
oof_f1 = [f1_score(y_train, oof_prob >= t) for t in thresholds]
best_threshold = float(thresholds[int(np.argmax(oof_f1))])
print('Selected threshold:', best_threshold)
print('OOF F1:', max(oof_f1))


## 7. Final held-out test evaluation
Apply the selected threshold once to the held-out test probabilities. Report metrics that reflect the rare failure class.


In [ ]:
pred_tuned = (prob_rf >= best_threshold).astype(int)
print('Confusion matrix:\n', confusion_matrix(y_test, pred_tuned))
print(classification_report(y_test, pred_tuned, digits=3))
print('ROC-AUC:', roc_auc_score(y_test, prob_rf))
print('PR-AUC:', average_precision_score(y_test, prob_rf))


## 8. Feature importance
Impurity-based feature importance is useful for exploration but does not establish causal effects.


In [ ]:
feature_names = model_rf.named_steps['pre'].get_feature_names_out()
importances = model_rf.named_steps['clf'].feature_importances_
importance_df = (pd.DataFrame({'feature': feature_names, 'importance': importances})
                 .sort_values('importance', ascending=False))
importance_df.head(15)


## 9. Exploratory analysis: torque and tool wear
These plots are descriptive and do not establish causal relationships.


In [ ]:
eda = df.copy()
eda['Status'] = eda['Machine failure'].map({0: 'Healthy', 1: 'Failure'})
fig = px.scatter(eda, x='Torque [Nm]', y='Tool wear [min]', color='Status',
                 title='Torque vs Tool wear by machine status', opacity=0.6)
fig.show()


## Notes and limitations
- The target is imbalanced, so accuracy alone can be misleading.
- The threshold is selected to maximize F1 on training out-of-fold predictions; deployment thresholds should reflect operational costs.
- Test metrics are specific to this dataset and split and do not guarantee performance on another factory or time period.
- Impurity-based feature importance has limitations; permutation importance or SHAP can provide complementary analysis.
- Torque × tool wear is an exploratory proxy, not a validated failure mechanism.
